# Bài toán 3 — Thiết lập và xây dựng mô hình

## 5. Thiết lập thực nghiệm

Dùng ba lượt validation theo thời gian để so sánh. Trong mỗi lượt chỉ dùng dữ liệu trước ngày bắt đầu validation để tính hệ số và dự báo đủ 548 ngày. Giữ riêng 548 ngày cuối cho đánh giá sau khi chọn phương pháp.

Cần `statsmodels` trong kernel Python. Nếu chưa có, cài bằng `%pip install statsmodels` trong một ô Jupyter rồi chạy lại notebook.

In [4]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sales_path = Path("data/raw/sales.csv")
start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT", Path.cwd())).resolve()
project_root = next(
    (folder for folder in (start, *start.parents) if (folder / sales_path).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/sales.csv. Hãy đặt notebook trong dự án "
        "hoặc trỏ RETAIL_ANALYTICS_ROOT về thư mục gốc."
    )

targets = ["Revenue", "COGS"]
columns = ["Date", *targets]
sales_daily = pd.read_csv(project_root / sales_path, usecols=columns, dtype="string")[columns]
sales_daily["Date"] = pd.to_datetime(sales_daily["Date"], format="%Y-%m-%d", errors="raise")
if sales_daily["Date"].isna().any():
    raise ValueError("Date có giá trị thiếu.")

sales_daily[targets] = sales_daily[targets].apply(pd.to_numeric, errors="raise")
sales_daily = sales_daily.sort_values("Date", ignore_index=True)

history_start = pd.Timestamp("2012-07-04")
history_end = pd.Timestamp("2022-12-31")
expected_dates = pd.date_range(history_start, history_end, freq="D")
if not pd.DatetimeIndex(sales_daily["Date"]).equals(expected_dates):
    raise ValueError("Lịch sử có ngày thiếu, trùng hoặc ngoài khoảng đã xác nhận.")
if not np.isfinite(sales_daily[targets].to_numpy(dtype=float)).all():
    raise ValueError("Revenue hoặc COGS có giá trị thiếu hoặc không hữu hạn.")


horizon = 548
n_folds = 3
dev_start = history_start
dev_end = sales_daily["Date"].iloc[-horizon - 1]
sales_dev = sales_daily.set_index("Date").loc[dev_start:dev_end].copy()



one_day = pd.Timedelta(days=1)

test_dates = sales_daily.loc[sales_daily["Date"] > dev_end, "Date"]
if len(test_dates) != horizon:
    raise ValueError("Tập test phải có đúng 548 ngày.")

rows = []
for fold in range(1, n_folds + 1):
    val_end = dev_end - pd.Timedelta(days=(n_folds - fold) * horizon)
    val_start = val_end - pd.Timedelta(days=horizon - 1)
    train_end = val_start - one_day

    train = sales_dev.loc[:train_end]
    validation = sales_dev.loc[val_start:val_end]
    if train.empty or len(validation) != horizon:
        raise ValueError(f"Fold {fold} không đủ dữ liệu.")

    rows.append({
        "Fold": fold,
        "Train từ": train.index.min().date(),
        "Train đến": train_end.date(),
        "Số ngày train": len(train),
        "Validation từ": val_start.date(),
        "Validation đến": val_end.date(),
        "Số ngày validation": len(validation),
    })

folds = pd.DataFrame(rows)
display(folds)
print(f"Test cuối: {test_dates.min():%d/%m/%Y}–{test_dates.max():%d/%m/%Y} ({len(test_dates)} ngày)")
assert (folds["Train đến"] < folds["Validation từ"]).all()
assert pd.to_datetime(folds["Validation đến"]).max() < test_dates.min()

,Fold,Train từ,Train đến,Số ngày train,Validation từ,Validation đến,Số ngày validation
0,1,2012-07-04,2016-12-30,1641,2016-12-31,2018-07-01,548
1,2,2012-07-04,2018-07-01,2189,2018-07-02,2019-12-31,548
2,3,2012-07-04,2019-12-31,2737,2020-01-01,2021-07-01,548


Test cuối: 02/07/2021–31/12/2022 (548 ngày)


## 6. Xây dựng nhiều phương pháp

Mỗi phương pháp được định nghĩa bên dưới và chạy qua cùng ba lượt validation ở mục 6.10. Revenue và COGS được khớp riêng trong các phương pháp cần ước lượng tham số. Notebook lưu dự báo cho từng lượt để đánh giá ở notebook 03.

In [5]:
import sys
from time import perf_counter
import warnings

import statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | NumPy {np.__version__} | statsmodels {statsmodels.__version__}")

contest_name = "Baseline cuộc thi"
fixed_name = "Mùa vụ, mức gốc cố định"
mean_name = "Trung bình 28 ngày"
selected_label = "Phương pháp được chọn"


def repeat_level(level, dates):
    return pd.DataFrame(
        np.tile(level.to_numpy(dtype=float), (len(dates), 1)),
        index=dates, columns=level.index,
    )

Python 3.11.7 | pandas 2.2.3 | NumPy 2.4.6 | statsmodels 0.15.0


### 6.1. Naive

Lặp lại giá trị cuối cùng của tập train cho toàn bộ kỳ dự báo. Đây là mốc đơn giản nhất.

In [6]:
def predict_naive(train, dates):
    return repeat_level(train.iloc[-1], dates)

### 6.2. Trung bình 28 ngày

Dùng trung bình của 28 ngày cuối tập train làm mức dự báo cố định.

In [7]:
def predict_mean_28(train, dates):
    return repeat_level(train.tail(28).mean(), dates)

### 6.3. Theo thứ trong tuần

Dùng trung bình ứng với từng thứ trong 28 ngày cuối. Thứ của ngày dự báo được xác định từ lịch.

In [8]:
def predict_weekday(train, dates):
    recent = train.tail(28)
    weekday = recent.groupby(recent.index.dayofweek).mean()
    forecast = weekday.reindex(dates.dayofweek)
    forecast.index = dates
    return forecast

### 6.4. ETS (A,N,N)

Làm trơn mũ đơn giản để ước lượng mức của chuỗi. Mô hình dùng sai số cộng, không có thành phần xu hướng hoặc mùa vụ.

In [9]:
def predict_ets(train, dates):
    forecast = pd.DataFrame(index=dates, columns=train.columns, dtype=float)
    for target in train.columns:
        history = train[target].astype(float).asfreq("D") / 1_000_000
        fitted = SimpleExpSmoothing(history, initialization_method="estimated").fit()
        estimate = fitted.forecast(len(dates))
        pd.testing.assert_index_equal(estimate.index, dates, check_names=False)
        forecast[target] = estimate.to_numpy(dtype=float) * 1_000_000
    return forecast

### 6.5. ARIMA(1,1,1)

Lấy sai phân một lần, sau đó khớp thành phần tự hồi quy bậc 1 và trung bình trượt bậc 1. Dùng cùng cấu hình cho từng chỉ tiêu và từng lượt.

In [10]:
def predict_arima(train, dates):
    forecast = pd.DataFrame(index=dates, columns=train.columns, dtype=float)
    for target in train.columns:
        history = train[target].astype(float).asfreq("D") / 1_000_000
        with warnings.catch_warnings():
            # Statsmodels dùng điểm khởi tạo bằng 0 khi ước lượng ban đầu không phù hợp.
            warnings.filterwarnings(
                "ignore", message=r"Non-(stationary|invertible) starting .*", category=UserWarning,
            )
            fitted = ARIMA(history, order=(1, 1, 1)).fit()
        if not fitted.mle_retvals["converged"]:
            raise RuntimeError(f"ARIMA không hội tụ cho {target}, mốc {train.index[-1]:%d/%m/%Y}.")
        estimate = fitted.forecast(len(dates))
        pd.testing.assert_index_equal(estimate.index, dates, check_names=False)
        forecast[target] = estimate.to_numpy(dtype=float) * 1_000_000
    return forecast

### 6.6. Hồi quy tuyến tính

Dùng số ngày kể từ ngày đầu tập train làm đặc trưng và khớp một đường xu hướng riêng cho từng chỉ tiêu.

In [11]:
def predict_trend(train, dates):
    origin = train.index[0]
    train_days = (train.index - origin).days.to_numpy()
    future_days = (dates - origin).days.to_numpy()
    slope, intercept = np.polyfit(train_days, train.to_numpy(dtype=float), 1)
    forecast = pd.DataFrame(
        intercept + future_days[:, None] * slope, index=dates, columns=train.columns,
    )
    return forecast

### 6.7. Hồi quy tuyến tính + mùa vụ

Tính phần dư của đường xu hướng trên tập train, lấy trung bình theo cặp tháng–ngày rồi cộng vào dự báo tương ứng.

In [12]:
def predict_trend_with_seasonality(train, dates):
    trend = predict_trend(train, train.index.append(dates))
    residuals = train - trend.loc[train.index]
    seasonal = residuals.groupby([train.index.month, train.index.day]).mean()
    month_day = pd.MultiIndex.from_arrays([dates.month, dates.day])
    adjustment = seasonal.reindex(month_day).fillna(0.0)
    adjustment.index = dates
    return trend.loc[dates] + adjustment

### 6.8. Baseline cuộc thi

Dùng mức ngày của năm đầy đủ gần nhất, hệ số tăng trưởng hằng năm và hệ số mùa vụ theo tháng–ngày. Tất cả được tính lại từ tập train của từng lượt.

In [13]:
def predict_contest_baseline(train, forecast_dates):
    years = train.index.year
    days_observed = train.groupby(years).size()
    days_in_year = pd.to_datetime(
        days_observed.index.astype(str) + "-12-31"
    ).dayofyear
    complete_years = days_observed.index[days_observed.eq(days_in_year)]

    # Tính mức gốc và tăng trưởng từ các năm đủ dữ liệu.
    annual_totals = train.groupby(years).sum().loc[complete_years]
    base_year = complete_years[-1]
    base_level = annual_totals.loc[base_year] / days_observed.loc[base_year]
    growth = (annual_totals.iloc[-1] / annual_totals.iloc[0]) ** (
        1 / (base_year - complete_years[0])
    )

    # Tính hệ số mùa vụ theo cặp tháng–ngày từ tập train.
    normalized = train.div(train.groupby(years).transform("mean"))
    seasonal = normalized.groupby([train.index.month, train.index.day]).mean()
    month_day = pd.MultiIndex.from_arrays([forecast_dates.month, forecast_dates.day])
    seasonal_factors = seasonal.reindex(month_day).fillna(1.0)
    seasonal_factors.index = forecast_dates

    years_ahead = (forecast_dates.year - base_year).to_numpy()
    growth_factors = growth.to_numpy() ** years_ahead[:, None]
    return seasonal_factors.mul(growth_factors).mul(base_level).round(2)

### 6.9. Mùa vụ, mức gốc cố định

Giữ mức trung bình ngày của năm đầy đủ gần nhất, sau đó nhân với hệ số mùa vụ theo tháng–ngày học từ các quan sát trong tập train.

In [14]:
def predict_fixed_level_seasonal(train, forecast_dates):
    # Lấy trung bình ngày của năm đầy đủ gần nhất làm mức gốc.
    years = train.index.year
    days_observed = train.groupby(years).size()
    days_in_year = pd.to_datetime(
        days_observed.index.astype(str) + "-12-31"
    ).dayofyear
    complete_years = days_observed.index[days_observed.eq(days_in_year)]
    base_year = complete_years[-1]
    base_level = train.loc[str(base_year)].mean()

    # Tính hệ số mùa vụ cho từng cặp tháng–ngày từ tập train.
    normalized = train.div(train.groupby(years).transform("mean"))
    seasonal = normalized.groupby([train.index.month, train.index.day]).mean()
    month_day = pd.MultiIndex.from_arrays([forecast_dates.month, forecast_dates.day])
    factors = seasonal.reindex(month_day).fillna(1.0)
    factors.index = forecast_dates

    return factors.mul(base_level).round(2)

### 6.10. Chạy các phương pháp và lưu dự báo thực nghiệm

So sánh MAE trung bình qua ba lượt validation để chọn phương pháp cho từng chỉ tiêu. Sau khi chọn, tạo dự báo tập test cho phương pháp được chọn và baseline cuộc thi. Bảng lưu gồm dự báo, mốc train và tên phương pháp. Thời gian chạy ở đây là thời gian khớp và dự báo trung bình theo lượt.

In [15]:
predictors = {
    "Naive": predict_naive,
    "Trung bình 28 ngày": predict_mean_28,
    "Theo thứ trong tuần": predict_weekday,
    "ETS (A,N,N)": predict_ets,
    "ARIMA(1,1,1)": predict_arima,
    "Hồi quy tuyến tính": predict_trend,
    "Hồi quy tuyến tính + mùa vụ": predict_trend_with_seasonality,
    "Baseline cuộc thi": predict_contest_baseline,
    "Mùa vụ, mức gốc cố định": predict_fixed_level_seasonal,
}

def evaluate(actual, forecast):
    pd.testing.assert_index_equal(actual.index, forecast.index)
    pd.testing.assert_index_equal(actual.columns, forecast.columns)
    if not np.isfinite(forecast.to_numpy(dtype=float)).all():
        raise ValueError("Dự báo có giá trị thiếu hoặc không hữu hạn.")
    actual = actual.astype(float) / 1_000_000
    error = forecast.astype(float) / 1_000_000 - actual
    squared_error = error.pow(2)
    variation = (actual - actual.mean()).pow(2).sum()
    return pd.DataFrame({
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt(squared_error.mean()),
        "R²": 1 - squared_error.sum() / variation.where(variation > 0),
    }).rename_axis("Chỉ tiêu")

from tempfile import TemporaryDirectory

prediction_parts = []
score_rows = []
timing_rows = []


def prediction_block(forecast, split, fold_id, train_end, method):
    return forecast.rename_axis("Date").reset_index().assign(
        Split=split, Fold=fold_id, TrainEnd=train_end, Method=method,
    )


for _, fold in folds.iterrows():
    fold_id = int(fold["Fold"])
    train_end = pd.Timestamp(fold["Train đến"])
    train = sales_dev.loc[:train_end, targets]
    actual = sales_dev.loc[pd.Timestamp(fold["Validation từ"]):pd.Timestamp(fold["Validation đến"]), targets]

    for method, predictor in predictors.items():
        started = perf_counter()
        forecast = predictor(train, actual.index)
        timing_rows.append({"Phương pháp": method, "Giây/lượt": perf_counter() - started})
        score_rows.append(evaluate(actual, forecast).assign(Method=method))
        prediction_parts.append(prediction_block(forecast, "validation", fold_id, train_end, method))

mean_mae = pd.concat(score_rows).reset_index().groupby(["Method", "Chỉ tiêu"])["MAE"].mean()
selected_methods = mean_mae.unstack("Chỉ tiêu").idxmin().reindex(targets)
display(selected_methods.rename("Phương pháp được chọn theo MAE"))
display(pd.DataFrame(timing_rows).groupby("Phương pháp").mean().reindex(predictors).round(3))

final_train = sales_dev[targets]
final_dates = pd.DatetimeIndex(test_dates)
assert final_train.index.max() + pd.Timedelta(days=1) == final_dates.min()
reference_forecast = predictors[contest_name](final_train, final_dates)
selected_forecast = pd.DataFrame(index=final_dates, columns=targets, dtype=float)
for method in selected_methods.unique():
    forecast = predictors[method](final_train, final_dates)
    chosen_targets = selected_methods.index[selected_methods.eq(method)]
    selected_forecast[chosen_targets] = forecast[chosen_targets]

prediction_parts.extend([
    prediction_block(reference_forecast, "test", 0, final_train.index.max(), contest_name),
    prediction_block(selected_forecast, "test", 0, final_train.index.max(), selected_label),
])
predictions = pd.concat(prediction_parts, ignore_index=True)[
    ["Split", "Fold", "TrainEnd", "Date", "Method", *targets]
]

prediction_path = project_root / "data/model/du_bao/forecast_predictions.csv"
prediction_path.parent.mkdir(parents=True, exist_ok=True)
csv_text = predictions.to_csv(index=False, date_format="%Y-%m-%d", lineterminator=chr(10))
if not prediction_path.is_file() or prediction_path.read_text(encoding="utf-8") != csv_text:
    with TemporaryDirectory(dir=prediction_path.parent) as directory:
        temporary_path = Path(directory) / prediction_path.name
        temporary_path.write_bytes(csv_text.encode("utf-8"))
        temporary_path.replace(prediction_path)
print(f"Đã xác nhận {len(predictions):,} dòng dự báo tại {prediction_path.relative_to(project_root)}.")

Chỉ tiêu
Revenue    Mùa vụ, mức gốc cố định
COGS       Mùa vụ, mức gốc cố định
Name: Phương pháp được chọn theo MAE, dtype: object

,Giây/lượt
Phương pháp,
Naive,0.001
Trung bình 28 ngày,0.001
Theo thứ trong tuần,0.006
"ETS (A,N,N)",0.063
"ARIMA(1,1,1)",0.735
Hồi quy tuyến tính,0.001
Hồi quy tuyến tính + mùa vụ,0.011
Baseline cuộc thi,0.018
"Mùa vụ, mức gốc cố định",0.017


Đã xác nhận 15,892 dòng dự báo tại data\model\du_bao\forecast_predictions.csv.
